<a href="https://colab.research.google.com/github/macin-tx/extrator-pln/blob/main/notebooks/extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Instalação das bibliotecas e do modelo linguístico


In [ ]:
!pip install -U pypdf spacy openpyxl
!python -m spacy download pt_core_news_sm

# 2. Conexão com o google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Monta os caminhos
from pathlib import Path
pasta_projeto = Path('/content/drive/MyDrive/extractor')
pasta_pdfs = pasta_projeto / 'pdfs'
pasta_resultados = pasta_projeto / 'resultados'

# cria os diretórios
pasta_pdfs.mkdir(parents=True, exist_ok=True)
pasta_resultados.mkdir(parents=True, exist_ok=True)

# lista os pdfs inseridos
for pdf in pasta_pdfs.glob("*.pdf"):
  print(pdf.name)

# 3. Interface

In [ ]:
#@title Formulário de busca
arquivo = "Capitulo5.pdf" #@param {type:"string"}
busca_por = "termos" #@param ["termos", "classes gramaticais"]
termos = "ontologia, corpus, doença" #@param {type:"string"}
classes = "NOUN, PROPN, ADJ" #@param {type:"string"}

# lematização na busca permite contar plural e singular juntas
usar_lema = True #@param {type:"boolean"}
formato = "csv" #@param ["xlsx", "csv"]

# converter os campos em listas
lista_termos = [t.strip().lower() for t in termos.split(',') if t.strip()]
lista_classes = [c.strip().upper() for c in classes.split(',') if c.strip()]

print("Termos:", lista_termos)
print("Classes:", lista_classes)




# 4. Leitura do PDF

In [88]:
from pypdf import PdfReader

# Definir o caminho do arquivo
caminho_pdf = pasta_pdfs / arquivo

# Verificar se o arquivo existe
if not caminho_pdf.is_file():
  raise FileNotFoundError(f"Arquivo não encontrado: {caminho_pdf}")


def extrair_texto(caminho):

  # Ler o pdf
  leitor = PdfReader(caminho)

  # Armazenar o numero da pagina e seu texto
  paginas = []
  # Armazenar apenas o conteudo textual dessas paginas
  textos = []

  # Percorrer as páginas do doc
  for numero, pagina in enumerate(leitor.pages, start=1):
    # Extrair o contéudo textual
    texto = pagina.extract_text() or ""

    # Desconsiderar páginas vazias
    if texto.strip():
      # Criar dicionário com duas chaves(n° da página e seu contéudo)
      paginas.append({"pagina": numero,
                      "texto": texto})

      # Adicionar apenas conteudo textual à lista textos
      textos.append(texto)

  # Interromper a execução caso nenhuma página tenha seu contéudo extraído
  if not paginas:
    raise ValueError("Nenhum texto foi extraído.")

  # Concatenar esses textos em uma única string separados por quebra de linha
  completo = "\n".join(textos)

  # Retornar lista de paginas e o texto completo
  return paginas, completo

paginas, texto_f = extrair_texto(caminho_pdf)

# print(f"Texto extraído: {len(texto_f)} caracteres em {len(paginas)} página(s) com texto")

# 5. Carregar modelo spaCy

In [ ]:
# Importar a biblioteca
import spacy

# Carregar o modelo
nlp = spacy.load("pt_core_news_sm")

def analisar_texto(paginas, usar_lema=True):
  tokens = []

  # Percorrer as páginas extraídas do PDF
  for p in paginas:

    # Recuperar o n° da pagina e seu conteudo
    numero = p["pagina"]
    texto = p["texto"]

    # Trocar quebra de linha por espaço
    texto = " ".join(texto.split())

    # Processar o texto
    doc = nlp(texto)

    # Iterar sobre os tokens
    for token in doc:

      if token.is_alpha: # Desconsiderar quando o token for espaço, numero e pontuacao
        tokens.append({
            "Pagina": numero,
            "Palavra": token.text.lower(),
            "Lema": token.lemma_.lower(),
            "Classe": token.pos_
        })
  return tokens

tokens = analisar_texto(paginas, usar_lema)
"""
print(f"{len(tokens)} tokens guardados")
for t in tokens[:10]:
  print(t)
"""


# 6. Executar busca e filtragem

In [90]:
# Definir a busca pela palavra original ou por seu lema
if usar_lema:
  atributo = "Lema"
else:
  atributo = "Palavra"

resultado = []

# Buscar por termo
if busca_por == "termos":
  if not lista_termos:
    raise ValueError("Nenhum termo informado.")

  # termos em letras minusculas
  lista_termos = [termo.lower().strip() for termo in lista_termos]

  for termo in lista_termos:
    quant = 0

    # Percorrer os tokens e verificar correspondência
    for token in tokens:
      if token[atributo].lower() == termo.lower():
        quant += 1

    # Registrar o termo com a quantidade em que ocorre
    resultado.append({
        "Termo": termo,
        "Ocorrência": quant
    })


# Buscar por classe gramatical
else:
  if not lista_classes:
    raise ValueError("Nenhuma classe gramatical informada.")

  # classes em letra maiuscula
  lista_classes = [classe.upper().strip() for classe in lista_classes]

  for classe in lista_classes:
    palavras = []

    # Identificar as palavras associadas às classes
    for token in tokens:
      if token["Classe"].upper() == classe.upper():
        palavras.append(token[atributo])

    # Remover duplicatas e ordenar
    palavras = sorted(set(palavras))

    # Associar cada palavra a sua classe
    for palavra in palavras:
      resultado.append({
          "Classe": classe.upper(),
          "Palavra": palavra
      })


# Exportação dos resultados

In [ ]:
import pandas as pd

# Converter o resultado em DataFrame
df = pd.DataFrame(resultado)

# Combinar o nome do pdf sem a extensao com tipo de busca realizado
nome_saida = f"{caminho_pdf.stem}_{busca_por}"

# Exportar resultados em xlsx ou cvs
if formato == "xlsx":
  df.to_excel(
      pasta_resultados / f"{nome_saida}.xlsx",
      index=False
  )
else:
  df.to_csv(
      pasta_resultados / f"{nome_saida}.csv",
      index=False,
      encoding="utf-8-sig",
      sep=";"
  )

display(df)
